# Paper 1 (Final/Best Version): ConvNeXtBase Multi-Modal Fusion Model
### Combines: Strongest CNN Family + Fine-Tuning + Class Weighting + Patient Metadata Fusion

**This notebook combines every improvement technique into one model:**
1. ConvNeXtBase (same family as the base paper's best model, ConvNeXtXLarge)
2. Fine-tuning (unfreezing last layers, not just frozen transfer learning)
3. Class-weighted loss (handles the dataset's imbalance)
4. Metadata fusion (age, sex, lesion location combined with image features) -- THIS IS YOUR NOVELTY

**IMPORTANT: Fresh Colab session, T4 GPU enabled.**
**Expected time: 90-120 minutes (ConvNeXtBase + 2-phase training is the longest combination so far).**

**Honest note:** This combination gives the best realistic chance of matching or exceeding the base paper's 87.62%, because it stacks every proven technique together. It is not guaranteed -- no ML result ever is -- but this is the strongest configuration possible within Colab's free-tier constraints.

## STEP 1: Kaggle Auth + Download Dataset

In [ ]:
import os
!mkdir -p ~/.kaggle
!echo "$KAGGLE_TOKEN" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!mkdir -p /content/ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/ham10000
print("Dataset ready!")

## STEP 2: Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import ConvNeXtBase
from tensorflow.keras.applications.convnext import preprocess_input as convnext_preprocess
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score, roc_auc_score

print("GPU Available:", tf.config.list_physical_devices('GPU'))

## STEP 3: Load Metadata + Clean It

In [ ]:
metadata = pd.read_csv('/content/ham10000/HAM10000_metadata.csv')
all_image_paths = glob.glob('/content/ham10000/**/*.jpg', recursive=True)
image_path_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in all_image_paths}
metadata['image_path'] = metadata['image_id'].map(image_path_dict.get)
metadata = metadata.dropna(subset=['image_path']).reset_index(drop=True)

metadata['age'] = metadata['age'].fillna(metadata['age'].median())
metadata['sex'] = metadata['sex'].fillna('unknown')
metadata['localization'] = metadata['localization'].fillna('unknown')

classes = sorted(metadata['dx'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
metadata['label_idx'] = metadata['dx'].map(class_to_idx)
print("Metadata loaded:", metadata.shape)

## STEP 4: Encode Metadata Features

In [ ]:
age_scaler = StandardScaler()
metadata['age_scaled'] = age_scaler.fit_transform(metadata[['age']])

sex_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
sex_encoded = sex_encoder.fit_transform(metadata[['sex']])

loc_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
loc_encoded = loc_encoder.fit_transform(metadata[['localization']])

metadata_features = np.concatenate([metadata[['age_scaled']].values, sex_encoded, loc_encoded], axis=1)
META_DIM = metadata_features.shape[1]
print("Metadata feature dimensions:", META_DIM)

for i in range(META_DIM):
    metadata[f'meta_{i}'] = metadata_features[:, i]
meta_cols = [f'meta_{i}' for i in range(META_DIM)]

## STEP 5: Train / Val / Test Split + Class Weights

In [ ]:
train_df, temp_df = train_test_split(metadata, test_size=0.3, stratify=metadata['dx'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['dx'], random_state=42)
print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)

class_weights_array = compute_class_weight(
    class_weight='balanced', classes=np.array(range(len(classes))), y=train_df['label_idx'].values
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights ready.")

## STEP 6: Custom Data Pipeline (Image + Metadata Together)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16  # smaller batch since ConvNeXtBase is a large model
NUM_CLASSES = len(classes)

def load_and_preprocess_image(path, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.2)
    img = convnext_preprocess(img)
    return img

def make_tf_dataset(df, augment=False, shuffle=False):
    paths = df['image_path'].values
    meta = df[meta_cols].values.astype(np.float32)
    labels = tf.keras.utils.to_categorical(df['label_idx'].values, NUM_CLASSES)

    def gen():
        for p, m, l in zip(paths, meta, labels):
            img = load_and_preprocess_image(p, augment=augment)
            yield (img, m), l

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            (tf.TensorSpec(shape=(224,224,3), dtype=tf.float32),
             tf.TensorSpec(shape=(META_DIM,), dtype=tf.float32)),
            tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32)
        )
    )
    if shuffle:
        ds = ds.shuffle(1000, seed=42)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_tf_dataset(train_df, augment=True, shuffle=True)
val_ds = make_tf_dataset(val_df, augment=False, shuffle=False)
test_ds = make_tf_dataset(test_df, augment=False, shuffle=False)
print("Datasets ready.")

## STEP 7: Build ConvNeXtBase + Metadata Fusion Model

In [ ]:
image_input = tf.keras.Input(shape=(224,224,3), name='image_input')
base_cnn = ConvNeXtBase(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_cnn.trainable = False
x_img = base_cnn(image_input, training=False)
x_img = layers.GlobalAveragePooling2D()(x_img)
x_img = layers.Dense(256, activation='relu')(x_img)
x_img = layers.Dropout(0.3)(x_img)

meta_input = tf.keras.Input(shape=(META_DIM,), name='meta_input')
x_meta = layers.Dense(32, activation='relu')(meta_input)
x_meta = layers.Dropout(0.2)(x_meta)
x_meta = layers.Dense(16, activation='relu')(x_meta)

fused = layers.Concatenate()([x_img, x_meta])
x = layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(fused)
x = layers.Dropout(0.3)(x)
output = layers.Dense(NUM_CLASSES, activation='softmax')(x)

fusion_model = models.Model(inputs=[image_input, meta_input], outputs=output)
fusion_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                      loss='categorical_crossentropy', metrics=['accuracy'])
print("Total params:", fusion_model.count_params())

## STEP 8: PHASE 1 - Train Head Only (Frozen ConvNeXtBase). Expected: ~30-40 min

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7)

history1 = fusion_model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    callbacks=[early_stop, reduce_lr], class_weight=class_weights, verbose=1
)

## STEP 9: PHASE 2 - Fine-Tune ConvNeXtBase. Expected: ~45-60 min

In [ ]:
base_cnn.trainable = True
for layer in base_cnn.layers[:-20]:
    layer.trainable = False

fusion_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                      loss='categorical_crossentropy', metrics=['accuracy'])

early_stop2 = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr2 = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-8)

history2 = fusion_model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    callbacks=[early_stop2, reduce_lr2], class_weight=class_weights, verbose=1
)

fusion_model.save('/content/convnext_fusion_final.keras')
print("Model saved!")

## STEP 10: Final Evaluation

In [ ]:
y_pred_probs = fusion_model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_df['label_idx'].values

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
y_true_bin = label_binarize(y_true, classes=list(range(len(classes))))
auc = roc_auc_score(y_true_bin, y_pred_probs, average='macro', multi_class='ovr')

print("="*60)
print("FINAL MODEL: ConvNeXtBase + Fine-Tuning + Class Weights + Metadata Fusion")
print("="*60)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"AUC:       {auc:.4f}")
print(f"\nVs. your ResNet50 baseline (78.24%): {(acc-0.7824)*100:+.2f} points")
print(f"Vs. base paper's best model, ConvNeXtXLarge (87.62%): {(acc-0.8762)*100:+.2f} points")

## STEP 11: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix - ConvNeXtBase Fusion Model (Final)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('/content/final_model_confusion_matrix.png', dpi=300)
plt.show()
print(classification_report(y_true, y_pred, target_names=classes))

## STEP 12: Full Comparison Table (Everything So Far)

In [ ]:
final_table = pd.DataFrame({
    'ResNet50 (baseline, frozen)': {'Accuracy': 0.7824, 'F1-Score': 0.5427, 'AUC': 0.9365},
    'DenseNet121 (baseline, frozen)': {'Accuracy': 0.7745, 'F1-Score': 0.4794, 'AUC': 0.9310},
    'MobileNetV2 (baseline, frozen)': {'Accuracy': 0.7638, 'F1-Score': 0.5166, 'AUC': 0.9182},
    'EfficientNetB0 (baseline, frozen)': {'Accuracy': 0.7585, 'F1-Score': 0.4809, 'AUC': 0.9254},
    'InceptionV3 (baseline, frozen)': {'Accuracy': 0.7525, 'F1-Score': 0.4370, 'AUC': 0.9132},
    'VGG16 (baseline, frozen)': {'Accuracy': 0.7425, 'F1-Score': 0.4268, 'AUC': 0.8868},
    'Proposed: ConvNeXtBase + Fine-Tune + Class-Weight + Metadata Fusion': {
        'Accuracy': acc, 'F1-Score': f1, 'AUC': auc
    }
}).T
final_table = final_table.sort_values('Accuracy', ascending=False)
print(final_table.round(4))
final_table.to_csv('/content/complete_final_comparison.csv')

## STEP 13: Download Everything

In [ ]:
from google.colab import files
files.download('/content/complete_final_comparison.csv')
files.download('/content/final_model_confusion_matrix.png')

---
## Your Novelty Statement (For the Paper)

> "While prior work, including our reference base paper, relies solely on image-based CNN classification with frozen transfer learning, we propose a hybrid multi-modal fusion framework that (1) fine-tunes a ConvNeXtBase backbone rather than using frozen features, (2) applies class-weighted loss to address the severe class imbalance in HAM10000, and (3) fuses patient metadata (age, sex, and anatomical location) with visual features to mirror real dermatological decision-making. This combined approach achieves [X]% accuracy, compared to 78.24% for our image-only ResNet50 baseline and 87.62% for the base paper's best (but substantially larger) ConvNeXtXLarge model."

**This framing works even if your accuracy doesn't exceed 87.62%** -- because your contribution is not "a bigger number" but "a smarter, more clinically realistic, and more efficient approach." That is a legitimate and common way papers get accepted.